# Fit a classifier without leaking the test set

Prerequisites: Vectors, conditional probability, gradients, and the distinction between fitting and evaluation.

A binary logistic model assigns p(y=1|x)=sigmoid(wᵀx+b). Cross-entropy penalizes confident wrong predictions; its gradient averages x(p−y). Starting with zero weights predicts 1/2 for every example. A useful first check is whether training loss falls after a modest update, not whether the model looks sophisticated.

The companion generates 180 fictional two-feature observations with seed 7 and splits them into 100 training, 40 validation and 40 test examples before fitting anything. The scaler learns its mean and standard deviation from training data only. Even unsupervised preprocessing can leak information if it sees held-out observations. The test split is inspected after the training choices in this fixed demonstration; repeated tuning against it would invalidate its role.

Confusion-matrix rows are true classes and columns are predicted classes. Compare the model with the training-majority baseline. Accuracy on 40 synthetic test cases is evidence about this fixture, not a production estimate. To extend the experiment, create a separate evaluation cohort with a changed boundary and inspect which errors increase.

## Model structure

| Element | Model | Interpretation |
| --- | --- | --- |
| Training | 100 rows; fit scaler and weights | Estimate parameters |
| Validation | 40 rows | Compare choices before freezing them |
| Test | 40 untouched rows | Evaluate the frozen procedure |

## Predict

At zero weights, p=0.5 and loss=−log(0.5)≈0.6931. After 300 steps with learning rate 0.2, the notebook must show lower training loss. The confusion matrix must contain exactly 40 held-out examples.

In [ ]:
from pathlib import Path
import sys, math
base=Path.cwd().parent if Path.cwd().name=='notebooks' else Path.cwd()
sys.path.insert(0,str(base))
import ai_models as m

In [ ]:
result=m.classifier()
assert abs(result['initial_loss']-math.log(2))<1e-12
assert result['final_loss']<result['initial_loss']
assert sum(map(sum,result['test']['confusion']))==40
print(result)

## Perturb and explain

Change the data-generating coefficient or noise level in a copied function. Freeze the trained model, then measure the resulting error slice without refitting the scaler on the new cohort.

Record the changed input, prediction, result and a limit of your conclusion. Restart the kernel to reset the experiment.

## Check your reasoning

Where should the scaler learn its mean?

1. All 180 rows
2. Training rows only
3. Test rows only

Answer and explanations: 1: That lets held-out distribution information enter fitting. 2: The fitted preprocessing is part of the training procedure. 3: That directly conditions preprocessing on evaluation data.

## Primary evidence

- [scikit-learn — Common pitfalls: data leakage](https://scikit-learn.org/stable/common_pitfalls.html#data-leakage)

Review scope: this authored worked example and its stated assumptions; no hardware or production acceptance is implied.